In [2]:
import pandas as pd
import numpy as np

from xgboost import XGBRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.linear_model import PoissonRegressor


In [3]:
historical_df = pd.read_csv("historical_features.csv")

In [4]:
wc_group_stage = pd.read_csv("wc_group_stage_features.csv")

In [5]:
features = [
    'home_scored',
    'home_conceded',
    'home_win_rate',
    'away_scored',
    'away_conceded',
    'away_win_rate',
    'neutral',
    'tournament_weight',
    'h2h_home_wins',
    'h2h_away_wins',
    'h2h_draws',
    'elo_home',
    'elo_away',
    'elo_diff',
    'home_draw_rate',
    'away_draw_rate',
    'home_goal_diff',
    'away_goal_diff'
]

X = historical_df[features]

y_home = historical_df["home_score"]
y_away = historical_df["away_score"]

In [6]:
train = historical_df[historical_df["year"] < 2022]
test = historical_df[historical_df["year"] >= 2022]

In [7]:
X_train = train[features]
X_test = test[features]

y_home_train = train["home_score"]
y_home_test = test["home_score"]

y_away_train = train["away_score"]
y_away_test = test["away_score"]

In [8]:
from sklearn.preprocessing import StandardScaler

score_scaler = StandardScaler()

X_train_scaled = score_scaler.fit_transform(X_train)
X_test_scaled = score_scaler.transform(X_test)

In [9]:
home_goal_model = PoissonRegressor(
    alpha=0.01,
    max_iter=2000
)

away_goal_model = PoissonRegressor(
    alpha=0.50,
    max_iter=2000
)

home_goal_model.fit(
    X_train_scaled,
    y_home_train
)

away_goal_model.fit(
    X_train_scaled,
    y_away_train
)

,"alpha alpha: float, default=1Constant that multiplies the L2 penalty term and determines theregularization strength. ``alpha = 0`` is equivalent to unpenalizedGLMs. In this case, the design matrix `X` must have full column rank(no collinearities).Values of `alpha` must be in the range `[0.0, inf)`.",0.5
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the linear predictor (`X @ coef + intercept`).",True
,"solver solver: {'lbfgs', 'newton-cholesky'}, default='lbfgs'Algorithm to use in the optimization problem:'lbfgs' Calls scipy's L-BFGS-B optimizer.'newton-cholesky' Uses Newton-Raphson steps (in arbitrary precision arithmetic equivalent to iterated reweighted least squares) with an inner Cholesky based solver. This solver is a good choice for `n_samples` >> `n_features`, especially with one-hot encoded categorical features with rare categories. Be aware that the memory usage of this solver has a quadratic dependency on `n_features` because it explicitly computes the Hessian matrix. .. versionadded:: 1.2",'lbfgs'
,"max_iter max_iter: int, default=100The maximal number of iterations for the solver.Values must be in the range `[1, inf)`.",2000
,"tol tol: float, default=1e-4Stopping criterion. For the lbfgs solver,the iteration will stop when ``max{|g_j|, j = 1, ..., d} <= tol``where ``g_j`` is the j-th component of the gradient (derivative) ofthe objective function.Values must be in the range `(0.0, inf)`.",0.0001
,"warm_start warm_start: bool, default=FalseIf set to ``True``, reuse the solution of the previous call to ``fit``as initialization for ``coef_`` and ``intercept_`` .",False
,"verbose verbose: int, default=0For the lbfgs solver set verbose to any positive number for verbosity.Values must be in the range `[0, inf)`.",0


In [10]:
home_goal_pred = home_goal_model.predict(X_test_scaled)
away_goal_pred = away_goal_model.predict(X_test_scaled)

In [11]:
home_mae = mean_absolute_error(
    y_home_test,
    home_goal_pred
)

away_mae = mean_absolute_error(
    y_away_test,
    away_goal_pred
)

home_rmse = np.sqrt(
    mean_squared_error(
        y_home_test,
        home_goal_pred
    )
)

away_rmse = np.sqrt(
    mean_squared_error(
        y_away_test,
        away_goal_pred
    )
)

print(f"Home Goals MAE: {home_mae:.3f}")
print(f"Away Goals MAE: {away_mae:.3f}")

print(f"Home Goals RMSE: {home_rmse:.3f}")
print(f"Away Goals RMSE: {away_rmse:.3f}")

Home Goals MAE: 1.043
Away Goals MAE: 0.844
Home Goals RMSE: 1.366
Away Goals RMSE: 1.124


In [12]:
wc_X = wc_group_stage[features]
wc_X_scaled = score_scaler.transform(wc_X)

In [13]:
wc_group_stage["lambda_home"] = home_goal_model.predict(
    wc_X_scaled
)

wc_group_stage["lambda_away"] = away_goal_model.predict(
    wc_X_scaled
)

In [14]:
wc_group_stage[
    [
        "home_team",
        "away_team",
        "lambda_home",
        "lambda_away"
    ]
].head(10)

,home_team,away_team,lambda_home,lambda_away
0,Mexico,South Africa,2.437451,0.629158
1,South Korea,Czech Republic,1.379783,1.298805
2,Canada,Bosnia and Herzegovina,2.083089,0.687175
3,United States,Paraguay,1.535126,1.076766
4,Qatar,Switzerland,0.736311,2.265300
5,Brazil,Morocco,1.307862,1.405530
6,Haiti,Scotland,1.153947,1.547291
7,Australia,Turkey,1.330997,1.485552
8,Germany,Curaçao,3.103399,0.689424
9,Ivory Coast,Ecuador,0.933490,1.476770


In [15]:
print("Home λ range:",
      wc_group_stage["lambda_home"].min(),
      wc_group_stage["lambda_home"].max())

print("Away λ range:",
      wc_group_stage["lambda_away"].min(),
      wc_group_stage["lambda_away"].max())

Home λ range: 0.6302167194062234 3.264580553505103
Away λ range: 0.5176424835844277 2.6315182074700134


In [16]:
from scipy.stats import poisson

In [17]:
lambda_home = wc_group_stage.loc[0, "lambda_home"]
lambda_away = wc_group_stage.loc[0, "lambda_away"]

print("Home λ:", lambda_home)
print("Away λ:", lambda_away)

Home λ: 2.4374506811096532
Away λ: 0.6291583927082246


In [18]:
home_goals = np.arange(0, 6)

home_probabilities = poisson.pmf(
    home_goals,
    lambda_home
)

print(home_probabilities)

[0.08738334 0.21299257 0.25957944 0.21090403 0.12851704 0.06265079]


In [19]:
away_goals = np.arange(0, 6)

away_probabilities = poisson.pmf(
    away_goals,
    lambda_away
)

print(away_probabilities)

[5.33040223e-01 3.35366730e-01 1.05499396e-01 2.21252769e-02
 3.48007591e-03 4.37903793e-04]


In [20]:


def predict_goal_probabilities(fixtures,home_team, away_team, max_goals=5):

    # Find the match
    match = fixtures[
        (fixtures["home_team"] == home_team) &
        (fixtures["away_team"] == away_team)
    ]

    if match.empty:
        print(f"Match not found: {home_team} vs {away_team}")
        return

    # Get expected goals
    lambda_home = match.iloc[0]["lambda_home"]
    lambda_away = match.iloc[0]["lambda_away"]

    # Possible goals
    goals = np.arange(0, max_goals + 1)

    # Calculate probabilities
    home_probs = poisson.pmf(goals, lambda_home) * 100
    away_probs = poisson.pmf(goals, lambda_away) * 100

    # Most probable number of goals
    predicted_home_goals = goals[np.argmax(home_probs)]
    predicted_away_goals = goals[np.argmax(away_probs)]

    print(f"\n{home_team} vs {away_team}")
    print("-" * 40)

    print("\nExpected goals:")
    print(f"{home_team}: {lambda_home:.2f}")
    print(f"{away_team}: {lambda_away:.2f}")

    print("\nGoal probabilities:")

    for goal, home_prob, away_prob in zip(
        goals, home_probs, away_probs
    ):
        print(
            f"{goal} goals → "
            f"{home_team}: {home_prob:.2f}% | "
            f"{away_team}: {away_prob:.2f}%"
        )

    print("\nPredicted goals:")
    print(f"{home_team}: {predicted_home_goals}")
    print(f"{away_team}: {predicted_away_goals}")

    print(
        f"\nPredicted score: "
        f"{predicted_home_goals}-{predicted_away_goals}"
    )
        # Calculate exact scoreline probabilities
    scorelines = []

    for home_goal, home_prob in zip(goals, home_probs):
        for away_goal, away_prob in zip(goals, away_probs):

            score_probability = (
                home_prob * away_prob / 100
            )

            scorelines.append(
                (
                    home_goal,
                    away_goal,
                    score_probability
                )
            )

    # Sort by probability
    scorelines.sort(
        key=lambda x: x[2],
        reverse=True
    )

    print("\nTop 5 most probable scorelines:")

    for home_goal, away_goal, probability in scorelines[:5]:

        print(
            f"{home_goal}-{away_goal}: "
            f"{probability:.2f}%"
        )

In [21]:
predict_goal_probabilities(wc_group_stage,"Mexico","South Africa")


Mexico vs South Africa
----------------------------------------

Expected goals:
Mexico: 2.44
South Africa: 0.63

Goal probabilities:
0 goals → Mexico: 8.74% | South Africa: 53.30%
1 goals → Mexico: 21.30% | South Africa: 33.54%
2 goals → Mexico: 25.96% | South Africa: 10.55%
3 goals → Mexico: 21.09% | South Africa: 2.21%
4 goals → Mexico: 12.85% | South Africa: 0.35%
5 goals → Mexico: 6.27% | South Africa: 0.04%

Predicted goals:
Mexico: 2
South Africa: 0

Predicted score: 2-0

Top 5 most probable scorelines:
2-0: 13.84%
1-0: 11.35%
3-0: 11.24%
2-1: 8.71%
1-1: 7.14%


In [22]:


def predict_scores(fixtures, max_goals=5):

    results = []

    for _, row in fixtures.iterrows():

        home_team = row["home_team"]
        away_team = row["away_team"]

        lambda_home = row["lambda_home"]
        lambda_away = row["lambda_away"]

        goals = np.arange(0, max_goals + 1)

        home_probs = poisson.pmf(goals, lambda_home)
        away_probs = poisson.pmf(goals, lambda_away)

        # Most probable individual goal count
        predicted_home_goals = goals[np.argmax(home_probs)]
        predicted_away_goals = goals[np.argmax(away_probs)]

        # Exact scoreline probabilities
        scorelines = []

        for hg, hp in zip(goals, home_probs):
            for ag, ap in zip(goals, away_probs):

                probability = hp * ap

                scorelines.append(
                    (hg, ag, probability)
                )

        scorelines.sort(
            key=lambda x: x[2],
            reverse=True
        )

        results.append({
            "home_team": home_team,
            "away_team": away_team,
            "expected_home_goals": round(lambda_home, 2),
            "expected_away_goals": round(lambda_away, 2),
            "predicted_home_goals": predicted_home_goals,
            "predicted_away_goals": predicted_away_goals,
            "predicted_score": f"{predicted_home_goals}-{predicted_away_goals}",
        
            "top_score_probability": round(scorelines[0][2] * 100, 2)
        })

    return pd.DataFrame(results)

In [23]:
predict_goal_probabilities(
    "Germany",
    "Curaçao"
)

TypeError: predict_goal_probabilities() missing 1 required positional argument: 'away_team'

In [ ]:
import joblib
import os

os.makedirs("models", exist_ok=True)

joblib.dump(home_goal_model, "models/home_goal_model.pkl")
joblib.dump(away_goal_model, "models/away_goal_model.pkl")
joblib.dump(score_scaler, "models/score_scaler.pkl")

print("Score models saved successfully.")

Score models saved successfully.


In [ ]:
r32_features=pd.read_csv("r32_features.csv")

In [ ]:
r32_features["lambda_home"] = home_goal_model.predict(
    score_scaler.transform(r32_features[features])
)

r32_features["lambda_away"] = away_goal_model.predict(
    score_scaler.transform(r32_features[features])
)

In [ ]:
predict_scores(r32_features)

,home_team,away_team,expected_home_goals,expected_away_goals,predicted_home_goals,predicted_away_goals,predicted_score,top_score_probability
0,South Africa,Canada,0.89,1.64,0,1,0-1,13.09
1,Brazil,Japan,1.39,1.24,1,1,1-1,12.44
2,Germany,Paraguay,1.93,0.99,1,0,1-0,10.40
3,Netherlands,Morocco,1.29,1.34,1,1,1-1,12.44
4,Ivory Coast,Norway,1.27,1.42,1,1,1-1,12.27
5,France,Sweden,2.69,0.82,2,0,2-0,10.84
6,Mexico,Ecuador,1.12,0.79,1,0,1-0,16.53
7,England,DR Congo,1.94,0.82,1,0,1-0,12.35
8,Belgium,Senegal,1.90,1.04,1,1,1-1,10.43
9,United States,Bosnia and Herzegovina,2.21,0.85,2,0,2-0,11.40


In [ ]:
predict_goal_probabilities(r32_features,"Spain", "Austria")

NameError: name 'predict_goal_probabilities' is not defined

In [ ]:
r16_features=pd.read_csv("r16_features.csv")
r16_features["lambda_home"] = home_goal_model.predict(
    score_scaler.transform(r16_features[features])
)

r16_features["lambda_away"] = away_goal_model.predict(
    score_scaler.transform(r16_features[features])
)

In [ ]:
predict_scores(r16_features)


,home_team,away_team,expected_home_goals,expected_away_goals,predicted_home_goals,predicted_away_goals,predicted_score,top_score_probability
0,Canada,Morocco,0.98,1.51,0,1,0-1,12.46
1,Paraguay,France,0.92,2.03,0,2,0-2,10.74
2,Brazil,Norway,1.78,1.12,1,1,1-1,10.95
3,Mexico,England,1.36,0.97,1,0,1-0,13.26
4,Portugal,Spain,0.92,1.39,0,1,0-1,13.80
5,United States,Belgium,1.31,1.46,1,1,1-1,11.97
6,Argentina,Egypt,2.24,0.75,2,0,2-0,12.57
7,Switzerland,Colombia,1.21,1.43,1,1,1-1,12.34


In [ ]:
predict_goal_probabilities(r16_features, "Mexico","England")


Mexico vs England
----------------------------------------

Expected goals:
Mexico: 1.36
England: 0.97

Goal probabilities:
0 goals → Mexico: 25.62% | England: 38.02%
1 goals → Mexico: 34.89% | England: 36.77%
2 goals → Mexico: 23.76% | England: 17.78%
3 goals → Mexico: 10.78% | England: 5.73%
4 goals → Mexico: 3.67% | England: 1.39%
5 goals → Mexico: 1.00% | England: 0.27%

Predicted goals:
Mexico: 1
England: 0

Predicted score: 1-0

Top 5 most probable scorelines:
1-0: 13.26%
1-1: 12.83%
0-0: 9.74%
0-1: 9.42%
2-0: 9.03%


In [ ]:
qf_features=pd.read_csv("qf_features.csv")
qf_features["lambda_home"] = home_goal_model.predict(
    score_scaler.transform(qf_features[features])
)

qf_features["lambda_away"] = away_goal_model.predict(
    score_scaler.transform(qf_features[features])
)

In [ ]:
predict_scores(qf_features)

,home_team,away_team,expected_home_goals,expected_away_goals,predicted_home_goals,predicted_away_goals,predicted_score,top_score_probability
0,France,Morocco,1.45,1.18,1,1,1-1,12.32
1,Spain,Belgium,1.74,1.05,1,1,1-1,11.24
2,Norway,England,1.13,1.56,1,1,1-1,11.96
3,Argentina,Switzerland,1.84,0.94,1,0,1-0,11.43


In [ ]:
predict_goal_probabilities(qf_features,"Norway","England")


Norway vs England
----------------------------------------

Expected goals:
Norway: 1.13
England: 1.56

Goal probabilities:
0 goals → Norway: 32.19% | England: 21.03%
1 goals → Norway: 36.49% | England: 32.79%
2 goals → Norway: 20.68% | England: 25.57%
3 goals → Norway: 7.81% | England: 13.29%
4 goals → Norway: 2.21% | England: 5.18%
5 goals → Norway: 0.50% | England: 1.62%

Predicted goals:
Norway: 1
England: 1

Predicted score: 1-1

Top 5 most probable scorelines:
1-1: 11.96%
0-1: 10.55%
1-2: 9.33%
0-2: 8.23%
1-0: 7.67%


In [ ]:
semi_features=pd.read_csv("semi_features.csv")
semi_features["lambda_home"] = home_goal_model.predict(
    score_scaler.transform(semi_features[features])
)

semi_features["lambda_away"] = away_goal_model.predict(
    score_scaler.transform(semi_features[features])
)

In [ ]:
predict_scores(semi_features)

,home_team,away_team,expected_home_goals,expected_away_goals,predicted_home_goals,predicted_away_goals,predicted_score,top_score_probability
0,France,Spain,1.04,1.30,1,1,1-1,13.02
1,England,Argentina,1.07,1.64,1,1,1-1,11.69


In [ ]:
predict_goal_probabilities(semi_features,"France","Spain")


France vs Spain
----------------------------------------

Expected goals:
France: 1.04
Spain: 1.30

Goal probabilities:
0 goals → France: 35.45% | Spain: 27.20%
1 goals → France: 36.76% | Spain: 35.41%
2 goals → France: 19.06% | Spain: 23.06%
3 goals → France: 6.59% | Spain: 10.01%
4 goals → France: 1.71% | Spain: 3.26%
5 goals → France: 0.35% | Spain: 0.85%

Predicted goals:
France: 1
Spain: 1

Predicted score: 1-1

Top 5 most probable scorelines:
1-1: 13.02%
0-1: 12.55%
1-0: 10.00%
0-0: 9.64%
1-2: 8.48%


In [ ]:
final_features=pd.read_csv("final_features.csv")
final_features["lambda_home"] = home_goal_model.predict(
    score_scaler.transform(final_features[features])
)

final_features["lambda_away"] = away_goal_model.predict(
    score_scaler.transform(final_features[features])
)

In [ ]:
predict_goal_probabilities(final_features,"Spain","Argentina")



Spain vs Argentina
----------------------------------------

Expected goals:
Spain: 1.26
Argentina: 1.32

Goal probabilities:
0 goals → Spain: 28.46% | Argentina: 26.81%
1 goals → Spain: 35.76% | Argentina: 35.29%
2 goals → Spain: 22.47% | Argentina: 23.23%
3 goals → Spain: 9.41% | Argentina: 10.19%
4 goals → Spain: 2.96% | Argentina: 3.35%
5 goals → Spain: 0.74% | Argentina: 0.88%

Predicted goals:
Spain: 1
Argentina: 1

Predicted score: 1-1

Top 5 most probable scorelines:
1-1: 12.62%
0-1: 10.04%
1-0: 9.59%
1-2: 8.31%
2-1: 7.93%
